# Real-Time Driver Drowsiness Detection System
## Exploratory Data Analysis (EDA) on Uploaded Datasets

This notebook performs thorough Exploratory Data Analysis (EDA) on the three Kaggle datasets physically uploaded to the project:
1. **Driver Drowsiness Dataset (DDD)**
2. **MRL Eye Dataset**
3. **Yawn Dataset**

### Objectives:
- Analyze dataset volume, class distributions, and balance ratios.
- Inspect resolutions, aspect ratios, and color spaces.
- Examine encoded metadata (subjects, glasses, lighting conditions).
- Detect data integrity issues (corrupted files, redundant duplicates).
- Map datasets to downstream deep learning tasks.

In [ ]:
import os
import sys
from pathlib import Path
from collections import Counter, defaultdict
import hashlib

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config
print(f"Project root resolved: {config.BASE_DIR}")
print(f"DDD Dataset: {config.DDD_DATASET_DIR.exists()}")
print(f"MRL Dataset: {config.MRL_DATASET_DIR.exists()}")
print(f"Yawn Dataset: {config.YAWN_DATASET_DIR.exists()}")

---
## 1. MRL Eye Dataset Analysis

The MRL Eye Dataset contains infrared close-up eye region images categorized into **Awake** (open eye) and **Sleepy** (closed eye).
Images are pre-partitioned into `train/`, `val/`, and `test/` directories.
Each filename encodes 8 attributes:
`s<subject_id>_<image_id>_<gender>_<glasses>_<eye_state>_<reflections>_<lighting>_<sensor_id>.png`

In [ ]:
mrl_path = config.MRL_DATASET_DIR
splits = ['train', 'val', 'test']
categories = ['awake', 'sleepy']

mrl_split_counts = defaultdict(dict)
total_awake = 0
total_sleepy = 0
mrl_dims = []

for sp in splits:
    for cat in categories:
        folder = mrl_path / sp / cat
        count = len(list(folder.glob('*.png')))
        mrl_split_counts[sp][cat] = count
        if cat == 'awake':
            total_awake += count
        else:
            total_sleepy += count

total_mrl = total_awake + total_sleepy
print(f"Total MRL Images: {total_mrl:,}")
print(f"  Open-Eye (Awake):   {total_awake:,} ({total_awake/total_mrl*100:.2f}%)")
print(f"  Closed-Eye (Sleepy): {total_sleepy:,} ({total_sleepy/total_mrl*100:.2f}%)")
print(f"  Class Balance Ratio: {total_awake/total_sleepy:.3f} : 1")
pd.DataFrame(mrl_split_counts).T

In [ ]:
# Display MRL Class Distribution and Sample Grid
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = ['Open-Eye (Awake)', 'Closed-Eye (Sleepy)']
counts = [total_awake, total_sleepy]
colors = ['#2b5c8f', '#e26d5c']
axes[0].bar(labels, counts, color=colors, width=0.5, edgecolor='black')
axes[0].set_title('MRL Eye Dataset: Class Distribution', fontweight='bold')
axes[0].set_ylabel('Number of Images')
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000, f"{bar.get_height():,}", ha='center', fontweight='bold')

split_df = pd.DataFrame(mrl_split_counts).T[['awake', 'sleepy']]
split_df.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_title('Distribution Across Splits', fontweight='bold')
axes[1].set_ylabel('Number of Images')
axes[1].legend(['Awake', 'Sleepy'])
plt.tight_layout()
plt.show()

---
## 2. Yawn Dataset Analysis

The Yawn Dataset contains facial crops of driver mouth and face regions classified into:
- `no yawn` (neutral/speaking mouth)
- `yawn` (extended open mouth yawn state)

Image resolutions vary across crops and represent extracted video frames.

In [ ]:
yawn_path = config.YAWN_DATASET_DIR
no_yawn_files = list((yawn_path / 'no yawn').glob('*.jpg'))
yawn_files = list((yawn_path / 'yawn').glob('*.jpg'))

count_no_yawn = len(no_yawn_files)
count_yawn = len(yawn_files)
total_yawn = count_no_yawn + count_yawn

print(f"Total Yawn Dataset Images: {total_yawn:,}")
print(f"  No-Yawn: {count_no_yawn:,} ({count_no_yawn/total_yawn*100:.2f}%)")
print(f"  Yawn:    {count_yawn:,} ({count_yawn/total_yawn*100:.2f}%)")
print(f"  Class Balance Ratio: {count_no_yawn/count_yawn:.3f} : 1")

# Dimension statistics
yawn_dims = [Image.open(p).size for p in (no_yawn_files + yawn_files)]
widths = [d[0] for d in yawn_dims]
heights = [d[1] for d in yawn_dims]
print(f"Resolution Range: Width ({min(widths)} to {max(widths)} px), Height ({min(heights)} to {max(heights)} px)")

In [ ]:
# Plot Yawn Class Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(['No-Yawn', 'Yawn'], [count_no_yawn, count_yawn], color=['#38b000', '#f77f00'], width=0.45, edgecolor='black')
axes[0].set_title('Yawn Dataset: Class Balance', fontweight='bold')
axes[0].set_ylabel('Number of Images')
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f"{bar.get_height():,}", ha='center', fontweight='bold')

sns.histplot(widths, color='#38b000', label='Width', kde=True, ax=axes[1])
sns.histplot(heights, color='#f77f00', label='Height', kde=True, ax=axes[1])
axes[1].set_title('Yawn Dataset: Crop Dimension Distribution', fontweight='bold')
axes[1].set_xlabel('Pixels')
axes[1].legend()
plt.tight_layout()
plt.show()

---
## 3. Driver Drowsiness Dataset (DDD) Analysis

The Driver Drowsiness Dataset (DDD) contains holistic driver camera viewpoints (face, posture, head pose).

### Inspection Findings:
- **Images vs Videos**: 41,793 images, 0 video files.
- **Classes**: Binary classes `Drowsy` (22,348) and `Non Drowsy` (19,445).
- **Subjects**: 26 identifiable recording streams denoted by letters (A-Z in Drowsy, a-z in Non Drowsy).
- **Day/Night & Glasses Categories**: NOT labeled in the uploaded dataset structure (no metadata or subfolder separation).
- **Resolution**: Fixed uniform 227x227 RGB images.

In [ ]:
ddd_path = config.DDD_DATASET_DIR
drowsy_files = list((ddd_path / 'Drowsy').glob('*.png'))
non_drowsy_files = list((ddd_path / 'Non Drowsy').glob('*.png'))

count_drowsy = len(drowsy_files)
count_non_drowsy = len(non_drowsy_files)
total_ddd = count_drowsy + count_non_drowsy

print(f"Total DDD Images: {total_ddd:,}")
print(f"  Drowsy:     {count_drowsy:,} ({count_drowsy/total_ddd*100:.2f}%)")
print(f"  Non-Drowsy: {count_non_drowsy:,} ({count_non_drowsy/total_ddd*100:.2f}%)")
print(f"  Balance Ratio: {count_drowsy/count_non_drowsy:.3f} : 1")

drowsy_subjects = Counter([f.name[0] for f in drowsy_files])
non_drowsy_subjects = Counter([f.name[0] for f in non_drowsy_files])
print(f"Identifiable Subjects/Streams: {len(drowsy_subjects)} streams (A-Z)")

In [ ]:
# Plot DDD Subject Distribution
fig, ax = plt.subplots(figsize=(14, 5))
letters = sorted(list(drowsy_subjects.keys()))
d_vals = [drowsy_subjects.get(l, 0) for l in letters]
nd_vals = [non_drowsy_subjects.get(l.lower(), 0) for l in letters]
x = np.arange(len(letters))
width = 0.38
ax.bar(x - width/2, d_vals, width, label='Drowsy (A-Z)', color='#e26d5c', edgecolor='black')
ax.bar(x + width/2, nd_vals, width, label='Non Drowsy (a-z)', color='#2b5c8f', edgecolor='black')
ax.set_title('DDD Dataset: Frames per Subject Identifier', fontweight='bold')
ax.set_xlabel('Subject Letter Prefix')
ax.set_ylabel('Number of Frames')
ax.set_xticks(x)
ax.set_xticklabels(letters)
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Key Takeaways & ML Task Assignment

| Dataset | Size | Nature of Images | Class Balance | Proposed ML Task in Pipeline |
| :--- | :--- | :--- | :--- | :--- |
| **MRL Eye Dataset** | 84,898 | Infrared eye crops (single eye, closed vs open) | 50.6% Awake / 49.4% Sleepy (Balanced) | **Eye State Classifier (PERCLOS & Blink Rate Analysis)** |
| **Yawn Dataset** | 5,119 | Mouth region crops (open mouth vs closed mouth) | 50.6% No-Yawn / 49.4% Yawn (Balanced) | **Yawn Detection Classifier (Mouth Aspect / Fatigue)** |
| **Driver Drowsiness Dataset (DDD)** | 41,793 | Full driver cabin / posture frames | 53.5% Drowsy / 46.5% Non-Drowsy | **Holistic Driver State & Nodding / Posture Classifier** |

### Architectural Synergy for Real-Time Detection:
1. **Facial Landmark Stage (MediaPipe)**: Detects face and extracts Region of Interest (ROI) for Eyes and Mouth.
2. **Specialized Deep Learning Models**:
   - **Eye Sub-Network** (trained on MRL): Real-time inference on Eye ROI for persistent eye closures (> threshold consecutive frames).
   - **Yawn Sub-Network** (trained on Yawn Dataset): Real-time inference on Mouth ROI for yawning episodes.
   - **Full-Frame / Holistic Sub-Network** (trained on DDD): Contextual backup validating head nod and driver posture fatigue.